# Support Vector Machines
**Chapter 5 — Hands-On ML**

Covers:
- Linear SVM Classification
- Non-linear SVM (Polynomial features + Polynomial Kernel + RBF Kernel)
- SVM Regression (LinearSVR + SVR with polynomial kernel)

Dataset: Iris (classification), make_moons (non-linear), synthetic (regression)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.svm import LinearSVC, SVC, LinearSVR, SVR
from sklearn.datasets import make_moons
import warnings
warnings.filterwarnings('ignore')

## 1. Linear SVM Classification

Finds the widest possible margin separating two classes. Only support vectors (boundary samples) influence the decision boundary.

In [ ]:
iris = datasets.load_iris()
X = iris['data'][:, (2, 3)]                    # petal length, width
y = (iris['target'] == 2).astype(float)        # 1 = Virginica

svm_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('linear_svc', LinearSVC(C=1, loss='hinge', random_state=42)),
])
svm_clf.fit(X, y)
print('Prediction for [5.5, 1.7]:', svm_clf.predict([[5.5, 1.7]]))

## 2. Non-Linear SVM Classification

Dataset: `make_moons` — two interleaving half-circles, not linearly separable.

In [ ]:
X_moons, y_moons = make_moons(n_samples=500, noise=0.30, random_state=42)

### 2a. Polynomial Features + LinearSVC

In [ ]:
poly_svm_clf = Pipeline([
    ('poly_features', PolynomialFeatures(degree=3)),
    ('scaler', StandardScaler()),
    ('linear_svc', LinearSVC(C=10, loss='hinge', random_state=42)),
])
poly_svm_clf.fit(X_moons, y_moons)
accuracy = poly_svm_clf.score(X_moons, y_moons)
print(f'Poly SVM accuracy: {accuracy:.3f}')

### 2b. Polynomial Kernel (kernel trick — no explicit feature expansion)

In [ ]:
poly_kernel_svm_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm_clf', SVC(kernel='poly', degree=3, coef0=1, C=5)),
])
poly_kernel_svm_clf.fit(X_moons, y_moons)
accuracy = poly_kernel_svm_clf.score(X_moons, y_moons)
print(f'Poly Kernel SVM accuracy: {accuracy:.3f}')

### 2c. RBF Kernel (Gaussian — similarity feature)

`gamma` controls how far each training sample's influence reaches. High gamma → tighter fit → overfitting risk.

In [ ]:
rbf_svm_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm_clf', SVC(kernel='rbf', gamma=5, C=0.001)),
])
rbf_svm_clf.fit(X_moons, y_moons)
accuracy = rbf_svm_clf.score(X_moons, y_moons)
print(f'RBF SVM accuracy: {accuracy:.3f}')

## 3. SVM Regression

Instead of maximizing margin between classes, SVR fits as many points as possible *within* the margin (epsilon-tube).

### 3a. Linear SVR

In [ ]:
np.random.seed(42)
X_reg = np.random.rand(100, 1)
y_reg = 4 * X_reg.ravel() + np.random.randn(100) + 3

svm_reg = LinearSVR(epsilon=1.5, random_state=42)
svm_reg.fit(X_reg, y_reg)
print('LinearSVR prediction at x=0.5:', svm_reg.predict([[0.5]]))

### 3b. Polynomial Kernel SVR

In [ ]:
svm_poly_reg = SVR(kernel='poly', degree=2, C=100, epsilon=0.1)
svm_poly_reg.fit(X_reg, y_reg)
print('SVR poly prediction at x=0.5:', svm_poly_reg.predict([[0.5]]))